# Kimi‑Linear (Gated DeltaNet‑2) — train a code LLM from scratch on Colab

This notebook runs the **Kimi‑Linear / GDN‑2** code‑generation pipeline
(JAX + Flax NNX) end‑to‑end on a free Colab **T4 GPU** — the exact hardware the
project is sized for.

**Pipeline:** train a byte‑level BPE tokenizer → build packed corpora from real
Hugging Face datasets → Stage‑1 pretrain → Stage‑2 anneal → generate code.

### Before you start
1. **Runtime → Change runtime type → Hardware accelerator: `T4 GPU`**, then Save.
2. Run the cells top to bottom.

> ⚠️ **A full run is long** (pretrain = 20k steps). This notebook has a
> `QUICK_DEMO` switch (Step 3) that shrinks the corpora and step counts so the
> whole pipeline finishes in ~15–25 min on a T4. Turn it off for a real run.

## 1. Check the GPU

In [1]:
!nvidia-smi

Thu Jul  2 21:25:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the project onto Colab

The repo isn't a public git repo yet, so pick **one** of the options below.
Option **A (Google Drive)** is the most reliable: upload the project folder to
your Drive once, then mount it every session.

### Option A — Google Drive (recommended)
Upload the whole `nugie-coding-llm-agent-3` folder to your Drive, then set `PROJECT_DIR` to its path.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Adjust this to wherever you put the folder in your Drive:
# PROJECT_DIR = '/content/drive/MyDrive/nugie-coding-llm-agent-3'

# import os
# assert os.path.isdir(PROJECT_DIR), f'Not found: {PROJECT_DIR} — upload the folder to Drive first.'
# os.chdir(PROJECT_DIR)
# print('Working in', os.getcwd())

### Option B — Upload a zip
Zip the project locally (`zip -r project.zip nugie-coding-llm-agent-3`) and upload it here.

In [ ]:
# from google.colab import files
# up = files.upload()                      # pick your project.zip
# import zipfile, os
# name = next(iter(up))
# with zipfile.ZipFile(name) as z:
#     z.extractall('/content')
# PROJECT_DIR = '/content/nugie-coding-llm-agent-3'   # adjust to the extracted folder name
# os.chdir(PROJECT_DIR)
# print('Working in', os.getcwd())

### Option C — git clone
If you push the project to GitHub, clone it here.

In [2]:
!git clone https://github.com/wisnunugroho21/nugie-coding-llm-agent-2.git /content/project
import os
PROJECT_DIR = '/content/project'
os.chdir(PROJECT_DIR)
print('Working in', os.getcwd())

Cloning into '/content/project'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 57 (delta 8), reused 55 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 58.45 KiB | 9.74 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Working in /content/project


## 3. Configuration

`QUICK_DEMO = True` shrinks everything so the full pipeline runs in minutes.
Set it to `False` to use the paper‑scale defaults from `configs/`.

In [3]:
QUICK_DEMO = True   # False = full T4 run (hours)
print('QUICK_DEMO =', QUICK_DEMO)

QUICK_DEMO = True


## 4. Install dependencies

Installs the pinned Python deps plus a CUDA‑12 JAX build for the T4 GPU.
Colab may prompt to **restart the runtime** after this — if it does, restart,
then **re‑run Step 2** (mount/chdir) and continue from Step 5.

In [4]:
!pip uninstall --yes jax jaxlib jax-cuda12-pjrt jax-cuda12-plugin jax-cuda13-pjrt jax-cuda13-plugin

Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: jax-cuda12-pjrt 0.7.2
Uninstalling jax-cuda12-pjrt-0.7.2:
  Successfully uninstalled jax-cuda12-pjrt-0.7.2
Found existing installation: jax-cuda12-plugin 0.7.2
Uninstalling jax-cuda12-plugin-0.7.2:
  Successfully uninstalled jax-cuda12-plugin-0.7.2


In [5]:
!pip install -q -r requirements.txt
# CUDA-13 JAX build matched to the Colab T4:
!pip install -q -U "jax[cuda13]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.1/525.1 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 129.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.4/124.4 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.7/184.7 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.7

## 5. Verify JAX sees the GPU

In [6]:
import jax
print('jax', jax.__version__)
print('devices:', jax.devices())
assert any(d.platform == 'gpu' for d in jax.devices()), \
    'No GPU visible to JAX. Set Runtime -> T4 GPU and reinstall jax[cuda13].'

jax 0.10.2
devices: [CudaDevice(id=0)]


## 6. Sanity tests

Confirms the GDN‑2 parallel training kernel matches the recurrent reference and
that the data pipeline wires up. Runs on CPU (fast, deterministic).

In [7]:
import os
env = dict(os.environ, JAX_PLATFORMS='cpu', PYTHONPATH=os.getcwd())
!JAX_PLATFORMS=cpu PYTHONPATH=$PWD python -m pytest tests -q

..                                                                       [100%]
2 passed in 68.88s (0:01:08)


## 7. (QUICK_DEMO) shrink the configs

Writes small overrides into the YAML configs so the full run finishes quickly:
tiny corpora, a small vocab, and short training. Skipped when `QUICK_DEMO=False`.

In [8]:
import yaml, pathlib

def patch_yaml(path, updates):
    p = pathlib.Path(path)
    d = yaml.safe_load(p.read_text())
    for keys, val in updates.items():
        node = d
        *parents, leaf = keys.split('.')
        for k in parents:
            node = node.setdefault(k, {})
        node[leaf] = val
    p.write_text(yaml.safe_dump(d, sort_keys=False))
    print('patched', path)

if QUICK_DEMO:
    patch_yaml('configs/data_pretrain.yaml', {
        'hf.max_documents': 2000,
    })
    patch_yaml('configs/data_anneal.yaml', {
        'hf.max_documents': 2000,
    })
    patch_yaml('configs/pretrain.yaml', {
        'train.max_steps': 200,
        'train.eval_every': 100,
        'train.ckpt_every': 100,
        'train.warmup_steps': 20,
        'train.log_every': 10,
    })
    patch_yaml('configs/anneal.yaml', {
        'train.max_steps': 100,
        'train.eval_every': 100,
        'train.ckpt_every': 50,
        'train.warmup_steps': 10,
        'train.log_every': 10,
    })
    # smaller vocab -> faster tokenizer training & smaller LM head
    VOCAB = 8000
    TOK_MAX_DOCS = 4000
else:
    VOCAB = 32000
    TOK_MAX_DOCS = 40000
print('vocab', VOCAB, 'tokenizer max-docs', TOK_MAX_DOCS)

patched configs/data_pretrain.yaml
patched configs/data_anneal.yaml
patched configs/pretrain.yaml
patched configs/anneal.yaml
vocab 8000 tokenizer max-docs 4000


## 8. Train the BPE tokenizer

Trains a from‑scratch byte‑level BPE (with FIM/EOT special tokens) on the
pretraining data source. First run downloads a slice of `bigcode/the-stack-smol`
from Hugging Face.

In [9]:
!PYTHONPATH=$PWD python -m training.data.train_tokenizer \
  --data-config configs/data_pretrain.yaml \
  --out tokenizers/code_bpe.json \
  --vocab-size $VOCAB \
  --max-docs $TOK_MAX_DOCS

usage: train_tokenizer.py [-h] --data-config DATA_CONFIG [--out OUT]
                          [--vocab-size VOCAB_SIZE]
                          [--min-frequency MIN_FREQUENCY]
                          [--max-docs MAX_DOCS]
train_tokenizer.py: error: argument --vocab-size: expected one argument


## 9. Build the packed corpora

Runs filter → dedup → tokenize → FIM → pack for both stages. Downloads the two
datasets from Hugging Face on first run.

In [10]:
# Stage-1 pretraining corpus  (bigcode/the-stack-smol, python)
!PYTHONPATH=$PWD python -m training.data.prepare --config configs/data_pretrain.yaml

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/project/training/data/prepare.py", line 162, in <module>
    main()
  File "/content/project/training/data/prepare.py", line 73, in main
    tok = CodeTokenizer(cfg.tokenizer)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/project/training/data/tokenizer.py", line 59, in __init__
    self.tok = Tokenizer.from_file(str(path))
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Exception: No such file or directory (os error 2)


In [11]:
# Stage-2 annealing corpus  (ise-uiuc/Magicoder-OSS-Instruct-75K)
!PYTHONPATH=$PWD python -m training.data.prepare --config configs/data_anneal.yaml

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/project/training/data/prepare.py", line 162, in <module>
    main()
  File "/content/project/training/data/prepare.py", line 73, in main
    tok = CodeTokenizer(cfg.tokenizer)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/project/training/data/tokenizer.py", line 59, in __init__
    self.tok = Tokenizer.from_file(str(path))
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Exception: No such file or directory (os error 2)


## 10. Stage‑1 — pretraining

Trains the hybrid GDN‑2 / MLA / MoE model. Logs loss, CE, MoE aux, LR and
tok/s; checkpoints land in `checkpoints/pretrain/`.

In [12]:
!PYTHONPATH=$PWD python -m training.train --config configs/pretrain.yaml

[config] ignoring unknown model keys: ['mlp_d_ff']
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/project/training/train.py", line 131, in <module>
    main()
  File "/content/project/training/train.py", line 53, in main
    meta = load_meta(tcfg.data_dir)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/project/training/data/loader.py", line 24, in load_meta
    with open(Path(data_dir) / "meta.json", "r") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/pretrain/meta.json'


## 11. Stage‑2 — annealing

Warm‑starts from the final pretrain checkpoint and decays the LR to zero on the
higher‑quality corpus. This cell auto‑detects the latest pretrain checkpoint.

In [13]:
import glob, os
ckpts = sorted(glob.glob('checkpoints/pretrain/model_*.msgpack'),
               key=lambda p: int(p.split('_')[-1].split('.')[0]))
assert ckpts, 'No pretrain checkpoint found — run Step 10 first.'
INIT = ckpts[-1]
print('warm-starting anneal from', INIT)
!PYTHONPATH=$PWD python -m training.train --config configs/anneal.yaml --init-from "$INIT"

AssertionError: No pretrain checkpoint found — run Step 10 first.

## 12. Generate code

Samples from the final annealed checkpoint.

In [ ]:
import glob
ckpts = sorted(glob.glob('checkpoints/anneal/model_*.msgpack'),
               key=lambda p: int(p.split('_')[-1].split('.')[0]))
assert ckpts, 'No anneal checkpoint found — run Step 11 first.'
MODEL = ckpts[-1]
print('generating from', MODEL)
!PYTHONPATH=$PWD python -m training.generate \
  --config configs/anneal.yaml --model "$MODEL" \
  --prompt "def quicksort(arr):" \
  --max-new-tokens 128 --temperature 0.8 --top-p 0.95

### Fill‑in‑the‑Middle (FIM) infill

In [ ]:
!PYTHONPATH=$PWD python -m training.generate \
  --config configs/anneal.yaml --model "$MODEL" \
  --prefix "def add(a, b):\n    return " \
  --suffix "\n\nprint(add(2, 3))"

## 13. Save checkpoints back to Drive

Colab wipes local disk on disconnect. This zips the checkpoints (plus the
tokenizer and packed‑corpus metadata) into a timestamped archive on your Drive
so a runtime reset doesn't lose the trained model.

If you used **Option A (Drive)** in Step 2, `PROJECT_DIR` already lives on Drive
and checkpoints are persisted as they're written — this cell still gives you a
single portable archive. If you used Option B/C (local disk), this is how you
get the model off the machine before it disconnects.

In [ ]:
import os, glob, datetime, shutil

# Where to drop the archive on Drive. Mount Drive (Step 2, Option A) if not already.
DRIVE_DIR = '/content/drive/MyDrive/kimi_linear_checkpoints'
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)

# Collect what's worth keeping: checkpoints, tokenizer, and per-corpus meta.json.
staging = '/content/_ckpt_bundle'
shutil.rmtree(staging, ignore_errors=True)
paths = (glob.glob('checkpoints/**/*.msgpack', recursive=True)
         + glob.glob('tokenizers/*.json')
         + glob.glob('data/**/meta.json', recursive=True))
assert paths, 'Nothing to save — run training first.'
for p in paths:
    dst = os.path.join(staging, p)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(p, dst)

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
base = os.path.join(DRIVE_DIR, f'kimi_linear_{stamp}')
archive = shutil.make_archive(base, 'zip', staging)
print('saved', archive, f'({os.path.getsize(archive)/1e6:.1f} MB)')
print('files:')
for p in paths:
    print('  ', p)

## Notes

- **QUICK_DEMO** produces a tiny, undertrained model — enough to see the full
  pipeline run and emit tokens, not to write real code. For a genuine model set
  `QUICK_DEMO=False` (Step 3) and expect a multi‑hour run; use Drive (Option A)
  so checkpoints survive disconnects.
- **Resume** an interrupted pretrain:
  `python -m training.train --config configs/pretrain.yaml --resume checkpoints/pretrain`
- **Scale the data** by editing `configs/data_*.yaml` (`hf.max_documents`, more
  language dirs, or a larger streaming source).
- **T4 precision:** the pipeline runs in fp32 (the T4 has no bf16 and the GDN‑2
  core needs fp32). See `README.md` for the fp16 path.